<a href="https://colab.research.google.com/github/BrenoBragaGalvao/mestrado-arvores-urbanas/blob/main/Segmentacao_area_completa_v2_experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V2 — comparar algoritmos e tamanhos de janela na Area1

O objetivo desta versão é **usar os dados existentes da Area1 para comparar oito classificadores de árvores e gramíneas, sem repetir o Quickshift global da imagem inteira**.

**Por quê?** A imagem tem **6.942 × 7.294 pixels**. Uma tentativa posterior de reproduzir o Quickshift global ultrapassou a memória disponível no ambiente de **4 GiB de RAM**; a memória da execução histórica original **não foi medida**. Por isso, utilizamos **janelas menores sem alterar a resolução dos pixels**.

**Há dois caminhos independentes neste notebook:**

| Caminho | O que faz | Como usar |
|---|---|---|
| **1. Comparar 512 × 512 e quatro partes de 256 × 256** | Repete o teste de **119 recortes**, duas segmentações, **oito algoritmos**, mesmas referências e partições; registra memória RAM | Execute **somente a próxima célula de código**, em uma sessão Google Colab com a Area1 no Drive |
| **2. Percorrer a imagem por quadrantes de 256 × 256** (opcional) | Percorre os **812 quadrantes** com checkpoints; prepara segmentos rotulados e compara **oito algoritmos** usando os mesmos registros e divisões na rota progressiva | Vá para a seção **Processamento progressivo de toda a grade**; execute Configuração → 1 → 2. As etapas 3 e 4, de proximidade e porte, são opcionais |

**Resultados já documentados, não executados por este notebook publicado:** na base derivada de 256 px, **SVM RBF** obteve F1-macro médio **0,8987** em blocos de 250 m (RF: **0,8656**). Na comparação das janelas, o SVM RBF chegou a **0,9341** com quatro 256 e blocos de 250 m. Os testes de memória registraram **467,19 MiB** (512 contínuo) e **456,80 MiB** (quatro 256), no ambiente medido. **Uma nova execução pode apresentar números diferentes; é necessário conferir os arquivos gerados com a documentação antes de alegar reprodução exata.**

**Importante:** 512 × 512 contínuo significa **cada recorte isolado**, não o Quickshift global original. A acurácia histórica **0,87** (92 exemplos de teste) não é comparável diretamente à F1-macro dos experimentos por janelas. **Não utilize “Executar tudo”**: as duas rotas são independentes, e o processamento dos 812 quadrantes é progressivo.

- [Resultados da comparação de oito algoritmos](docs/harmonizacao_protocolos_2026-11-01.md)
- [Resultados da comparação 512 vs. 256](docs/comparacao_controlada_segmentacao_512_vs_256_2026-11-01.md)
- [Medição do uso de RAM](docs/e9_medicao_memoria_2026-11-02.md)


## Caminho 1 — comparar os tamanhos de janela com oito algoritmos

Este experimento processa os **mesmos 119 recortes da Area1** de duas maneiras: **Quickshift em cada recorte inteiro de até 512 × 512 px** e **Quickshift em até quatro partes de 256 × 256 px**. Depois compara **oito classificadores**, usando os mesmos pontos e divisões espaciais. Os dois modos são executados em processos Python separados para medir o pico de RAM (RSS) de cada estratégia.

**Entrada:** somente os rasters e os pontos já existentes na pasta `Area1` do Google Drive. **Saídas:** tabelas de resultados, exemplos comuns, divisões, perfis de memória e relatório em `outputs/e9_reexecucao_v2`. São novos resultados de uma execução, não valores predefinidos.

**Uso:** execute apenas a célula abaixo para esse teste. Não precisa executar a rota progressiva ou o notebook original. Verifique se os arquivos `Area1/AOI_RGB_NEW.tif`, `Area1/AOI_NEW_MULTI.tif`, `Area1/Arvores.gpkg` e `Area1/Grama.gpkg` existem na pasta de Drive configurada.


In [ ]:
# CAMINHO 1 — RECORTES DE 512 × 512 VS QUATRO 256 × 256, OITO ALGORITMOS
# Esta célula lê os dados existentes e executa os dois modos com oito algoritmos.
# Recomenda-se uma sessão Colab nova com Drive montado.
%pip -q install rasterio geopandas shapely scikit-image scikit-learn scipy
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import urllib.request, subprocess, sys, json, pandas as pd
_AREA_E9=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1')
_OUT_E9=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/e9_reexecucao_v2')
_OUT_E9.mkdir(parents=True,exist_ok=True)
_SCRIPT_E9=Path('/content/e9_reproduzir_512_256.py')
urllib.request.urlretrieve(
  'https://raw.githubusercontent.com/BrenoBragaGalvao/mestrado-arvores-urbanas/main/experimentos/e9_reproduzir_512_256.py',
  str(_SCRIPT_E9))
for _modo_e9 in ('continuo_512','quadrantes_256','comparar'):
    print('Comparação de janelas | executando:',_modo_e9,flush=True)
    subprocess.run([sys.executable,str(_SCRIPT_E9),'--mode',_modo_e9,
                    '--area',str(_AREA_E9),'--out',str(_OUT_E9)],check=True)
print('Comparação de janelas | relatórios em:',_OUT_E9)
display(pd.read_csv(_OUT_E9/'resumo_oito_modelos.csv').round(4))
for _modo_e9 in ('continuo_512','quadrantes_256'):
    print('Pico RAM RSS:',_modo_e9,
          json.loads((_OUT_E9/f'{_modo_e9}_perfil.json').read_text())['pico_rss_mib'],'MiB')


## Caminho 2 (opcional) — percorrer a imagem por 812 quadrantes de 256 × 256 px

Este caminho é **independente** da comparação dos 119 recortes. Serve para percorrer a **grade inteira** com memória controlada: cada quadrante é segmentado separadamente, e os resultados intermediários são salvos no Drive para permitir continuar em outra sessão.

**Como executar:**
1. **Configuração:** monta o Drive, importa bibliotecas, encontra a Area1 e define as funções.
2. **Segmentar por lotes:** execute a próxima célula repetidamente, até concluir os 812 quadrantes (incluindo os de borda).
3. **Comparar oito algoritmos:** execute apenas após concluir a segmentação. Todos os oito modelos são avaliados sobre os **mesmos registros e divisões** quando usados os três atributos; as comparações de atributos isolados utilizam apenas os três modelos principais.
4. **Opcional — proximidade com postes e indicadores de porte:** execute as duas últimas células deste caminho, em sequência, somente se precisar dessa parte exploratória da pesquisa.

**Atenção:** esse processamento com quadrantes independentes não reproduz o Quickshift global nem o filtro global do cluster 3 de K-Means. Os resultados podem diferir dos números da bateria derivada publicada anteriormente. Não compare automaticamente os dois conjuntos.


In [ ]:
# CAMINHO 2 (OPCIONAL) — CONFIGURAÇÃO DOS 812 QUADRANTES
%pip -q install rasterio geopandas shapely scikit-image scikit-learn joblib
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import defaultdict
import os, json, gc, math, time, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window, transform as window_transform
from rasterio.features import shapes, geometry_mask
from skimage.segmentation import quickshift
from skimage.feature import canny
from skimage.color import rgb2hsv
from shapely.geometry import shape, box, mapping
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, silhouette_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import itertools, joblib
BASE_PROG = Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno')
AREA_PROG = BASE_PROG / 'Area1'
SAIDA_PROG = BASE_PROG / 'outputs' / 'v2_progressivo_256'
TILE_PROG = 256
TILES_POR_RODADA = 30  # Reduza para 10, se necessário.
DIR_TILES = SAIDA_PROG / 'tiles'
DIR_SPATIAL = SAIDA_PROG / 'spatial_tiles'
for pasta in [SAIDA_PROG, DIR_TILES, DIR_SPATIAL]:
    pasta.mkdir(parents=True, exist_ok=True)
for arquivo in ['AOI_RGB_NEW.tif','AOI_NEW_MULTI.tif','AOI_new.shp',
                'Arvores.gpkg','Grama.gpkg','postes.gpkg']:
    if not (AREA_PROG / arquivo).is_file():
        raise FileNotFoundError(f'Arquivo original ausente no Drive: {arquivo}')
with rasterio.open(AREA_PROG/'AOI_RGB_NEW.tif') as r:
    SHAPE_PROG = (r.height, r.width)
    TRANSFORM_PROG = r.transform
    CRS_PROG = r.crs
    RES_PROG = abs(r.res[0] * r.res[1])
with rasterio.open(AREA_PROG/'AOI_NEW_MULTI.tif') as r:
    if (r.height,r.width) != SHAPE_PROG or r.crs != CRS_PROG:
        raise ValueError('Rasters RGB e multiespectral sem dimensões/CRS compatíveis.')
    if not r.transform.almost_equals(TRANSFORM_PROG):
        warnings.warn('As duas grades raster apresentam deslocamento de origem; extração por janela é aproximação espacial e deve ser descrita.')
AOI_PROG = gpd.read_file(AREA_PROG/'AOI_new.shp').to_crs(CRS_PROG)
AOI_GEO_PROG = [mapping(g) for g in AOI_PROG.geometry if g is not None]
TILES_PROG = [(r,c) for r in range(0,SHAPE_PROG[0],TILE_PROG)
                    for c in range(0,SHAPE_PROG[1],TILE_PROG)]
REFERENCIAS_PROG = defaultdict(list)
for camada,classe in [('Arvores','arvore'),('Grama','grama')]:
    refs = gpd.read_file(AREA_PROG/f'{camada}.gpkg').to_crs(CRS_PROG)
    linhas,colunas = rasterio.transform.rowcol(TRANSFORM_PROG,refs.geometry.x.to_numpy(),refs.geometry.y.to_numpy())
    for linha,coluna in zip(linhas,colunas):
        if 0<=linha<SHAPE_PROG[0] and 0<=coluna<SHAPE_PROG[1]:
            REFERENCIAS_PROG[(int(linha)//TILE_PROG*TILE_PROG,
                              int(coluna)//TILE_PROG*TILE_PROG)].append((int(linha),int(coluna),classe))
def caminho_tile_prog(idx): return DIR_TILES / f'tile_{idx:04d}.npz'
def caminho_skip_prog(idx): return DIR_TILES / f'tile_{idx:04d}.skip'
def caminho_spatial_prog(idx): return DIR_SPATIAL / f'spatial_{idx:04d}.npz'
def salvar_npz_atomico_prog(destino, **arrays):
    temporario = destino.with_name(destino.name+'.tmp')
    with open(temporario,'wb') as f:
        np.savez_compressed(f,**arrays)
    os.replace(temporario,destino)
def um_tile_prog(idx):
    r0,c0 = TILES_PROG[idx]
    h,w = min(TILE_PROG,SHAPE_PROG[0]-r0),min(TILE_PROG,SHAPE_PROG[1]-c0)
    janela = Window(c0,r0,w,h)
    trans = window_transform(janela,TRANSFORM_PROG)
    dentro = geometry_mask(AOI_GEO_PROG,out_shape=(h,w),transform=trans,invert=True)
    if not dentro.any():
        caminho_skip_prog(idx).write_text('fora da AOI',encoding='utf-8')
        return 0
    with rasterio.open(AREA_PROG/'AOI_RGB_NEW.tif') as rgb:
        im_tile = np.moveaxis(rgb.read(window=janela),0,-1)
    with rasterio.open(AREA_PROG/'AOI_NEW_MULTI.tif') as mult:
        bandas = mult.read(window=janela)
    seg = quickshift(im_tile,kernel_size=3,max_dist=5,ratio=0.04).astype(np.int32)
    nseg = int(seg.max())+1
    flat,valid = seg.ravel(),dentro.ravel()
    count = np.bincount(flat[valid],minlength=nseg)
    def media(v):
        soma = np.bincount(flat[valid],weights=v.ravel()[valid],minlength=nseg)
        return np.divide(soma,count,out=np.full(nseg,np.nan),where=count>0)
    nir,red = bandas[3].astype(np.float32),bandas[0].astype(np.float32)
    ndvi = (nir-red)/(nir+red+1e-8)
    valor = rgb2hsv(np.moveaxis(bandas[:3],0,-1))[...,2].astype(np.float32)
    bordas = canny(nir/10000,sigma=1).astype(np.float32)
    feat3 = np.stack([media(valor),media(bordas),media(ndvi)],axis=1).astype(np.float32)
    oito = []
    for band in bandas:
        arr = band.astype(np.float64).ravel()
        ok = valid & (arr != 0)
        n = np.bincount(flat[ok],minlength=nseg)
        sx = np.bincount(flat[ok],weights=arr[ok],minlength=nseg)
        sx2 = np.bincount(flat[ok],weights=arr[ok]**2,minlength=nseg)
        m = np.divide(sx,n,out=np.full(nseg,np.nan),where=n>0)
        sd = np.sqrt(np.maximum(0,np.divide(sx2,n,out=np.full(nseg,np.nan),where=n>0)-m*m))
        oito.append(m);oito.append(sd)
    feat8 = np.column_stack([oito[i] for i in [0,2,4,6,1,3,5,7]]).astype(np.float32)
    rr,cc = np.indices((h,w))
    mr = media(rr.astype(np.float32))+r0+0.5
    mc = media(cc.astype(np.float32))+c0+0.5
    xs = TRANSFORM_PROG.c+TRANSFORM_PROG.a*mc+TRANSFORM_PROG.b*mr
    ys = TRANSFORM_PROG.f+TRANSFORM_PROG.d*mc+TRANSFORM_PROG.e*mr
    labels = []
    for r,c,classe in REFERENCIAS_PROG[(r0,c0)]:
        if dentro[r-r0,c-c0]:
            labels.append((int(seg[r-r0,c-c0]),0 if classe=='arvore' else 1))
    lseg = np.array([q[0] for q in labels],dtype=np.int32)
    lcls = np.array([q[1] for q in labels],dtype=np.int8)
    if nseg>65535: raise RuntimeError('Número inesperado de segmentos (>65535): aumente a representação dos IDs.')
    salvar_npz_atomico_prog(caminho_tile_prog(idx),seg=seg.astype(np.uint16),
        dentro=dentro.astype(np.uint8),count=count.astype(np.int32),
        f3=feat3,f8=feat8,x=xs.astype(np.float64),y=ys.astype(np.float64),
        lseg=lseg,lcls=lcls)
    del im_tile,bandas,seg,ndvi,bordas,valor
    gc.collect()
    return int(np.count_nonzero(count))
def executar_lote_prog(max_tiles=TILES_POR_RODADA):
    feitos=0
    for idx in range(len(TILES_PROG)):
        if caminho_tile_prog(idx).exists() or caminho_skip_prog(idx).exists(): continue
        n = um_tile_prog(idx)
        feitos += 1
        print(f'Quadrante {idx+1}/{len(TILES_PROG)}: {n} segmentos',flush=True)
        if feitos>=max_tiles: break
    prontos=sum(caminho_tile_prog(i).exists() or caminho_skip_prog(i).exists()
                for i in range(len(TILES_PROG)))
    print(f'\nCHECKPOINT: {prontos}/{len(TILES_PROG)} quadrantes concluídos.')
    if prontos<len(TILES_PROG): print('Repita a ETAPA A; recomeçará do próximo quadrante.')
    else: print('ETAPA A CONCLUÍDA: execute a ETAPA B.')
    return prontos==len(TILES_PROG)
print(f'Preparado: {len(TILES_PROG)} quadrantes, {SHAPE_PROG[0]}x{SHAPE_PROG[1]} pixels, {len(REFERENCIAS_PROG)} quadrantes com referências.')
print('Rota progressiva independente. Não execute as células históricas que carregam o raster inteiro.')

In [ ]:
# ETAPA 1 — SEGMENTAR POR LOTES (repetir até concluir os 812 quadrantes).
CONCLUIDO_A_PROG = executar_lote_prog(TILES_POR_RODADA)

In [ ]:
# ETAPA 2 — COMPARAR OITO CLASSIFICADORES, USANDO OS MESMOS DADOS E TESTES.
# Executar somente após terminar os 812 quadrantes da Etapa 1.
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import hashlib
if not all(caminho_tile_prog(i).exists() or caminho_skip_prog(i).exists()
           for i in range(len(TILES_PROG))):
    raise RuntimeError('A etapa A ainda não terminou. Repita sua célula até concluir.')
DSEG_PROG = SAIDA_PROG/'segmentos.csv.gz'
DREF_PROG = SAIDA_PROG/'referencias.csv'
if not DSEG_PROG.exists() or not DREF_PROG.exists():
    linhas,pares = [],[]
    for ti in range(len(TILES_PROG)):
        f=caminho_tile_prog(ti)
        if not f.exists(): continue
        with np.load(f,allow_pickle=False) as z:
            bons=np.where((z['count']>0)&np.isfinite(z['f3']).all(axis=1))[0]
            for si in bons:
                linhas.append((ti*65536+int(si),ti,int(si),int(z['count'][si]),
                   float(z['x'][si]),float(z['y'][si]),*map(float,z['f3'][si]),
                   *map(float,z['f8'][si])))
            for si,cl in zip(z['lseg'],z['lcls']):
                pares.append((ti*65536+int(si),'arvore' if cl==0 else 'grama'))
    nomes=['id','tile','sid','n_pixels','x','y','mean_intensity','mean_edges','mean_ndvi',
           'R_Mean','G_Mean','B_Mean','N_Mean','R_Std','G_Std','B_Std','N_Std']
    pd.DataFrame(linhas,columns=nomes).to_csv(DSEG_PROG,index=False,compression='gzip')
    pd.DataFrame(pares,columns=['id','classe']).to_csv(DREF_PROG,index=False)
    del linhas,pares
segmentos_prog=pd.read_csv(DSEG_PROG)
pares_prog=pd.read_csv(DREF_PROG)
features_prog=['mean_intensity','mean_edges','mean_ndvi']
unicos_prog=pares_prog.groupby('id')['classe'].nunique()
conflitos_prog=set(unicos_prog.index[unicos_prog>1])
rotulos_prog=(pares_prog.loc[~pares_prog.id.isin(conflitos_prog)]
              .drop_duplicates('id').merge(segmentos_prog,on='id',validate='one_to_one'))
rotulos_prog=rotulos_prog.replace([np.inf,-np.inf],np.nan).dropna(subset=features_prog)
rotulos_prog.to_csv(SAIDA_PROG/'rotulos_sem_duplicacao.csv',index=False)
print('E1 | rótulos brutos:',len(pares_prog),'únicos:',len(rotulos_prog),
      'conflitos:',len(conflitos_prog))
if len(rotulos_prog)<20 or rotulos_prog.classe.nunique()!=2:
    raise RuntimeError('Não há referências suficientes para as duas classes nos quadrantes.')
X_prog=rotulos_prog[features_prog].to_numpy(dtype=float)
y_prog=rotulos_prog.classe.to_numpy()
linhas_e1_prog=[]
for seed in [11,21,42,73,97]:
    tr,te=train_test_split(np.arange(len(y_prog)),test_size=.2,random_state=seed,stratify=y_prog)
    rf=RandomForestClassifier(n_estimators=500,random_state=42,n_jobs=2)
    rf.fit(X_prog[tr],y_prog[tr]); pred=rf.predict(X_prog[te])
    linhas_e1_prog.append({'semente':seed,'n_treino':len(tr),'n_teste':len(te),
       'f1_macro':f1_score(y_prog[te],pred,average='macro'),
       'acuracia_balanceada':balanced_accuracy_score(y_prog[te],pred)})
pd.DataFrame(linhas_e1_prog).to_csv(SAIDA_PROG/'E1_auditoria.csv',index=False)
auditoria_prog = {
  'referencias_brutas': int(len(pares_prog)),
  'segmentos_rotulados_distintos': int(pares_prog.id.nunique()),
  'registros_repetidos': int(len(pares_prog)-pares_prog.id.nunique()),
  'segmentos_conflitantes': int(len(conflitos_prog)),
  'segmentos_rotulados_validos': int(len(rotulos_prog)),
  'escopo': 'objetos de quadrantes independentes, nao segmentacao do raster integral'
}
(SAIDA_PROG/'E1_diagnostico.json').write_text(
    json.dumps(auditoria_prog,indent=2,ensure_ascii=False),encoding='utf-8')
# E2, E3, E4: blocos independentes, exclusão de centroides num raio de 25 m.
# Diferente do embargo geométrico por polígonos usado no código E2 original.
from scipy.spatial import cKDTree
resultados_prog=[];particoes_prog=[];descartadas_prog=[]
combos_prog=[c for k in [1,2,3] for c in itertools.combinations(features_prog,k)]
for tam in [250,500]:
    cx=rotulos_prog.x.to_numpy();cy=rotulos_prog.y.to_numpy()
    gx=np.floor((cx-cx.min())/tam).astype(int);gy=np.floor((cy-cy.min())/tam).astype(int)
    grupos=np.array([f'{a}_{b}' for a,b in zip(gx,gy)])
    if len(set(grupos))<5: continue
    for rodada,(tr,te) in enumerate(GroupShuffleSplit(n_splits=5,test_size=.2,random_state=42)
                                     .split(X_prog,y_prog,groups=grupos),1):
        distancia,_=cKDTree(np.column_stack([cx[te],cy[te]])).query(np.column_stack([cx[tr],cy[tr]]))
        tr=tr[distancia>25]
        if len(tr)<15 or len(te)<5 or len(set(y_prog[tr]))<2 or len(set(y_prog[te]))<2:
            descartadas_prog.append({'bloco_m':tam,'rodada':rodada,'motivo':'amostras insuficientes'})
            continue
        particoes_prog.append({'bloco_m':tam,'rodada':rodada,'n_treino':len(tr),'n_teste':len(te)})
        for combo in combos_prog:
            inds=[features_prog.index(s) for s in combo]; xx=X_prog[:,inds]
            # Três modelos básicos para a análise de atributos isolados.
            modelos={
                'Random Forest':RandomForestClassifier(n_estimators=500,random_state=42,n_jobs=2),
                'SVM RBF':make_pipeline(StandardScaler(),SVC(kernel='rbf')),
                'Regressao Logistica':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
            }
            # Com os três atributos OU a dupla fixa HSV+Canny, testar os oito modelos nas mesmas divisões.
            if len(combo)==3 or set(combo)=={'mean_intensity','mean_edges'}:
                modelos.update({
                    'KNN (7)':make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=7)),
                    'Extra Trees':ExtraTreesClassifier(n_estimators=300,random_state=42,n_jobs=2),
                    'HistGradientBoosting':HistGradientBoostingClassifier(random_state=42),
                    'Gaussian Naive Bayes':GaussianNB(),
                    'Dummy':DummyClassifier(strategy='most_frequent')
                })
            for nome,mdl in modelos.items():
                mdl.fit(xx[tr],y_prog[tr]); pp=mdl.predict(xx[te])
                resultados_prog.append({'bloco_m':tam,'rodada':rodada,'modelo':nome,
                  'atributos':'+'.join(combo),'n_treino':len(tr),'n_teste':len(te),
                  'f1_macro':f1_score(y_prog[te],pp,average='macro',zero_division=0),
                  'acuracia_balanceada':balanced_accuracy_score(y_prog[te],pp)})
        print('Comparação dos modelos | blocos:',tam,'rodada:',rodada,'concluída',flush=True)
pd.DataFrame(particoes_prog).to_csv(SAIDA_PROG/'E2_particoes.csv',index=False)
pd.DataFrame(descartadas_prog,columns=['bloco_m','rodada','motivo']).to_csv(SAIDA_PROG/'E2_descartadas.csv',index=False)
pd.DataFrame(resultados_prog).to_csv(SAIDA_PROG/'E3_E4_comparacoes.csv',index=False)
# E4 — síntese automática das sete combinações de atributos, sem reexecutar modelos.
# A evidência de cada rodada já está em E3_E4_comparacoes.csv.
e4_prog = pd.DataFrame(resultados_prog)
modelos_e4_prog = ['Random Forest','SVM RBF','Regressao Logistica']
e4_prog = e4_prog.loc[e4_prog.modelo.isin(modelos_e4_prog)].copy()
resumo_e4_prog = (e4_prog.groupby(['bloco_m','modelo','atributos'],as_index=False)
                  .agg(n_rodadas=('f1_macro','size'),
                       f1_macro_medio=('f1_macro','mean'),
                       f1_macro_dp=('f1_macro','std'),
                       acuracia_balanceada_media=('acuracia_balanceada','mean')))
resumo_e4_prog.to_csv(SAIDA_PROG/'E4_resumo_ablacao.csv',index=False)
atributos_completos_e4 = '+'.join(features_prog)
linhas_ganho_e4 = []
for (tam_e4,nome_e4),grupo_e4 in resumo_e4_prog.groupby(['bloco_m','modelo']):
    linha_full_e4 = grupo_e4.loc[grupo_e4.atributos.eq(atributos_completos_e4)]
    duplas_e4 = grupo_e4.loc[grupo_e4.atributos.str.count('\\+') == 1]
    if linha_full_e4.empty or duplas_e4.empty:
        continue
    melhor_dupla_e4 = duplas_e4.sort_values('f1_macro_medio',ascending=False).iloc[0]
    g_e4 = e4_prog.loc[(e4_prog.bloco_m==tam_e4)&(e4_prog.modelo==nome_e4)]
    comparacao_e4 = (g_e4.loc[g_e4.atributos.eq(atributos_completos_e4),['rodada','f1_macro']]
       .merge(g_e4.loc[g_e4.atributos.eq(melhor_dupla_e4.atributos),['rodada','f1_macro']],
              on='rodada',suffixes=('_tres','_dupla'),validate='one_to_one'))
    diff_e4 = comparacao_e4.f1_macro_tres-comparacao_e4.f1_macro_dupla
    linhas_ganho_e4.append(dict(bloco_m=tam_e4,modelo=nome_e4,
       melhor_dupla=melhor_dupla_e4.atributos,
       f1_tres=linha_full_e4.f1_macro_medio.iloc[0],
       f1_melhor_dupla=melhor_dupla_e4.f1_macro_medio,
       ganho_f1_medio=diff_e4.mean(),
       rodadas_com_ganho=int((diff_e4>0).sum()),
       rodadas_pareadas=len(diff_e4)))
ganhos_e4_prog=pd.DataFrame(linhas_ganho_e4)
ganhos_e4_prog.to_csv(SAIDA_PROG/'E4_tres_vs_melhor_dupla.csv',index=False)
print('E4 — média das sete combinações nos mesmos folds (resultados desta execução):')
print(resumo_e4_prog.sort_values(['bloco_m','modelo','f1_macro_medio'],
      ascending=[True,True,False]).round(4).to_string(index=False))
print('E4 — três atributos versus a melhor dupla em cada modelo/escala:')
print(ganhos_e4_prog.round(4).to_string(index=False))
print('A melhor dupla foi escolhida retrospectivamente: comparação descritiva, não teste externo.')
# Resumo legível: oito classificadores, somente os três atributos juntos.
df_modelos_256 = pd.DataFrame(resultados_prog)
comparacao_8_256 = df_modelos_256.loc[df_modelos_256['atributos'].eq('+'.join(features_prog))]
if comparacao_8_256.empty or comparacao_8_256['modelo'].nunique()!=8:
    raise RuntimeError('A comparação dos oito classificadores não foi concluída.')
resumo_8_256 = (comparacao_8_256.groupby(['bloco_m','modelo'])['f1_macro']
               .agg(['count','mean','std']).reset_index()
               .sort_values(['bloco_m','mean'],ascending=[True,False]))
resumo_8_256.to_csv(SAIDA_PROG/'comparacao_oito_algoritmos_256.csv',index=False)
print('\nOITO ALGORITMOS — F1-macro por escala de blocos (somente esta execução):')
print(resumo_8_256.round(4).to_string(index=False))

# E4b — Comparar a dupla fixa brilho HSV + bordas Canny contra os três atributos
# nos oito classificadores, usando os mesmos grupos e divisões do protocolo desta V2.
# Atenção: aqui o embargo é por centroides; a bateria publicada E3c usa geometria completa.
e4b_dupla_prog='mean_intensity+mean_edges'
e4b_tres_prog='+'.join(features_prog)
e4b_pares_prog=(df_modelos_256.loc[df_modelos_256.atributos.isin([e4b_dupla_prog,e4b_tres_prog])]
    .pivot(index=['bloco_m','rodada','modelo'],columns='atributos',values='f1_macro').reset_index())
if e4b_pares_prog[[e4b_dupla_prog,e4b_tres_prog]].isna().any().any():
    raise RuntimeError('E4b: alguma dupla ou trio não foi avaliado na mesma partição.')
if e4b_pares_prog.modelo.nunique()!=8:
    raise RuntimeError('E4b: esperados os oito classificadores.')
e4b_pares_prog['ganho_f1']=e4b_pares_prog[e4b_tres_prog]-e4b_pares_prog[e4b_dupla_prog]
e4b_resumo_prog=(e4b_pares_prog.groupby(['bloco_m','modelo'],as_index=False).agg(
   f1_tres=(e4b_tres_prog,'mean'),
   f1_hsv_canny=(e4b_dupla_prog,'mean'),
   ganho=('ganho_f1','mean'),
   rodadas_tres_superior=('ganho_f1',lambda x:int((x>0).sum())),
   n_rodadas=('ganho_f1','size')))
e4b_pares_prog.to_csv(SAIDA_PROG/'E4b_oito_modelos_rodadas.csv',index=False)
e4b_resumo_prog.to_csv(SAIDA_PROG/'E4b_oito_modelos_HSV_Canny_vs_tres.csv',index=False)
print('E4b — oito modelos; dupla fixa HSV+Canny vs. os três (resultados DESTA execução):')
print(e4b_resumo_prog.round(4).to_string(index=False))
print('E4b não replica automaticamente as médias E3c: diferenças nos embargos são documentadas.')

# RF completo para previsão exploratória por objeto; e clusters globais auxiliares.
rf_final_prog=RandomForestClassifier(n_estimators=500,random_state=42,n_jobs=2).fit(X_prog,y_prog)
joblib.dump(rf_final_prog,SAIDA_PROG/'rf_area1_lotes.joblib')
oito_prog=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std','G_Std','B_Std','N_Std']
limpos_prog=segmentos_prog.dropna(subset=oito_prog).copy()
amostra8_prog=limpos_prog[oito_prog].to_numpy(float)
clusters_e5_prog={}
resumos_e5_prog=[]
for k in [4,6,8]:
    mdl=KMeans(n_clusters=k,random_state=42,n_init=10)
    clusters_e5_prog[k]=mdl.fit_predict(amostra8_prog)
    resumos_e5_prog.append({'K':k,'total_segmentos':len(amostra8_prog),
       'silhouette_amostra_500':float(silhouette_score(amostra8_prog,clusters_e5_prog[k],sample_size=min(500,len(amostra8_prog)),random_state=42))})
    if k==6: joblib.dump(mdl,SAIDA_PROG/'kmeans6_area1_lotes.joblib')
pd.DataFrame(resumos_e5_prog).to_csv(SAIDA_PROG/'E5_kmeans.csv',index=False)
# E5a: cinco configurações Quickshift nas mesmas até quatro janelas de 256 px.
# Não modifica qualquer segmento salvo em A.
linhas_e5_qprog=[]
janelas_referencia_prog=list(REFERENCIAS_PROG.keys())[:4]
with rasterio.open(AREA_PROG/'AOI_RGB_NEW.tif') as src_e5:
    for indice_e5,(r0_e5,c0_e5) in enumerate(janelas_referencia_prog,1):
        w_e5=Window(c0_e5,r0_e5,min(256,SHAPE_PROG[1]-c0_e5),
                    min(256,SHAPE_PROG[0]-r0_e5))
        img_e5=np.moveaxis(src_e5.read(window=w_e5),0,-1)
        for ks_e5,md_e5,ra_e5 in [(2,5,.04),(3,3,.04),(3,5,.04),(3,8,.04),(4,5,.04)]:
            inicio_e5=time.perf_counter()
            s_e5=quickshift(img_e5,kernel_size=ks_e5,max_dist=md_e5,ratio=ra_e5)
            linhas_e5_qprog.append({'janela':indice_e5,'r0':r0_e5,'c0':c0_e5,
                'kernel_size':ks_e5,'max_dist':md_e5,'ratio':ra_e5,
                'n_segmentos':int(np.unique(s_e5).size),
                'tempo_s':time.perf_counter()-inicio_e5})
            del s_e5
pd.DataFrame(linhas_e5_qprog).to_csv(SAIDA_PROG/'E5_quickshift_piloto.csv',index=False)
print('Teste adicional de parâmetros Quickshift | execuções:',len(linhas_e5_qprog))
print('ETAPA 2 CONCLUÍDA. Resultados dos oito classificadores em:',SAIDA_PROG)
print('Estes resultados são da segmentação por quadrantes independentes, não da V2 integral original.')

In [ ]:
# ETAPA 3 (OPCIONAL) — proximidade a postes com checkpoints; repetir até concluir.
from shapely.strtree import STRtree
from shapely.geometry import Point
from rasterio.features import shapes
if not (SAIDA_PROG/'rf_area1_lotes.joblib').exists():
    raise RuntimeError('Execute a ETAPA B antes da proximidade espacial.')
RF_PROG = joblib.load(SAIDA_PROG/'rf_area1_lotes.joblib')
POSTES_PROG = (gpd.read_file(AREA_PROG/'postes.gpkg').to_crs(CRS_PROG)
               .clip(AOI_PROG).drop_duplicates(subset='geometry').reset_index(drop=True))
POSTES_SINDEX_PROG = POSTES_PROG.sindex
def um_spatial_prog(idx):
    origem=caminho_tile_prog(idx); destino=caminho_spatial_prog(idx)
    if not origem.exists():
        salvar_npz_atomico_prog(destino,seg_id=np.array([],dtype=np.int64),
           area=np.array([],dtype=np.float32),cls=np.array([],dtype=np.int8),
           par_seg=np.array([],dtype=np.int64),par_post=np.array([],dtype=np.int32),
           par_dist=np.array([],dtype=np.float32))
        return 0
    r0,c0=TILES_PROG[idx]; h=min(TILE_PROG,SHAPE_PROG[0]-r0);w=min(TILE_PROG,SHAPE_PROG[1]-c0)
    janela=Window(c0,r0,w,h)
    with np.load(origem,allow_pickle=False) as z:
        seg=z['seg'].astype(np.int32);dentro=z['dentro'].astype(bool)
        count=z['count'];f3=z['f3']
    ativos=np.flatnonzero((count>0)&np.isfinite(f3).all(axis=1))
    area=count[ativos]*RES_PROG
    clas=np.where(RF_PROG.predict(f3[ativos])=='arvore',0,1).astype(np.int8)
    ids=(idx*65536+ativos).astype(np.int64)
    # Todos os pares objeto–poste até 5 m, contando segmentos e postes distintos depois.
    geom_tile=box(*rasterio.windows.bounds(janela,TRANSFORM_PROG))
    prox_post=POSTES_SINDEX_PROG.query(geom_tile.buffer(5.5))
    pares_seg=[];pares_post=[];pares_dist=[]
    if len(prox_post):
        # Poligonizar apenas objetos que entram em ao menos um dos 12 cenários E7 > 5.
        params=[(.5,.7),(.6,.8),(2.5,.4),(4.,.5),(.4,.9),(1.35,.72)]
        possiveis=np.zeros(len(ativos),dtype=bool)
        for a,b in params:
            possiveis|=((a*np.power(area,b)>5)|(a+area*b>5))
        elegiveis=ativos[possiveis]
        if len(elegiveis):
            masc=dentro & np.isin(seg,elegiveis)
            for raw,id_segmento in shapes(seg,mask=masc,transform=window_transform(janela,TRANSFORM_PROG)):
                si=int(id_segmento)
                poly=shape(raw)
                candidatos=POSTES_SINDEX_PROG.query(poly.buffer(5.001))
                for p in candidatos:
                    d=poly.distance(POSTES_PROG.geometry.iloc[int(p)])
                    if d<=5.000001:
                        pares_seg.append(idx*65536+si)
                        pares_post.append(int(p))
                        pares_dist.append(float(d))
    salvar_npz_atomico_prog(destino,seg_id=ids,area=area.astype(np.float32),cls=clas,
       par_seg=np.asarray(pares_seg,dtype=np.int64),
       par_post=np.asarray(pares_post,dtype=np.int32),
       par_dist=np.asarray(pares_dist,dtype=np.float32))
    del seg,dentro,f3;gc.collect()
    return len(pares_seg)
novos=0
for ti in range(len(TILES_PROG)):
    if caminho_spatial_prog(ti).exists(): continue
    n=um_spatial_prog(ti);novos+=1
    print(f'Espacial {ti+1}/{len(TILES_PROG)}: {n} pares objeto–poste',flush=True)
    if novos>=TILES_POR_RODADA: break
feitos=sum(caminho_spatial_prog(i).exists() for i in range(len(TILES_PROG)))
print(f'CHECKPOINT ESPACIAL: {feitos}/{len(TILES_PROG)} quadrantes.')
print('ETAPA C CONCLUÍDA: execute D.' if feitos==len(TILES_PROG) else 'Repita C para continuar do último quadrante.')

In [ ]:
# ETAPA 4 (OPCIONAL) — consolidar proximidade e indicadores de porte.
# Execute somente depois de completar a Etapa 3.
if not all(caminho_spatial_prog(i).exists() for i in range(len(TILES_PROG))):
    raise RuntimeError('Etapa C incompleta: repita a célula C até todos os quadrantes serem processados.')
seg_linhas=[];par_linhas=[]
for ti in range(len(TILES_PROG)):
    with np.load(caminho_spatial_prog(ti),allow_pickle=False) as z:
        seg_linhas.extend(zip(z['seg_id'].tolist(),z['area'].tolist(),z['cls'].tolist()))
        par_linhas.extend(zip(z['par_seg'].tolist(),z['par_post'].tolist(),z['par_dist'].tolist()))
seg_df_prog=pd.DataFrame(seg_linhas,columns=['id','area','classe_cod']).drop_duplicates('id')
pares_df_prog=pd.DataFrame(par_linhas,columns=['id','poste_id','distancia_m'])
if not pares_df_prog.empty:
    pares_df_prog=pares_df_prog.groupby(['id','poste_id'],as_index=False)['distancia_m'].min()
seg_df_prog.to_csv(SAIDA_PROG/'segmentos_preditos.csv.gz',index=False,compression='gzip')
pares_df_prog.to_csv(SAIDA_PROG/'pares_segmento_poste_5m.csv.gz',index=False,compression='gzip')
# E6: seleção C3 > 7 sem e com filtro previsto como árvore, e raios de 1,3,5 metros.
seg_df_prog['C3'] = 2.5*np.power(seg_df_prog['area'],.4)
linhas_e6=[]
for filtro in ['todos','arvore']:
    escolha=seg_df_prog.loc[(seg_df_prog.C3>7)&
             ((seg_df_prog.classe_cod==0) if filtro=='arvore' else True),'id']
    for raio in [1,3,5]:
        proximos=pares_df_prog.loc[(pares_df_prog.distancia_m<=raio)&
                                    (pares_df_prog.id.isin(escolha))]
        linhas_e6.append({'filtro':filtro,'distancia_m':raio,
            'segmentos_elegiveis':len(escolha),'segmentos_proximos':proximos.id.nunique(),
            'postes_proximos':proximos.poste_id.nunique()})
pd.DataFrame(linhas_e6).to_csv(SAIDA_PROG/'E6_postes.csv',index=False)
# E7: 12 expressões × 3 limiares × 2 filtros × 3 distâncias = 216 cenários.
params_prog=[(.5,.7),(.6,.8),(2.5,.4),(4.,.5),(.4,.9),(1.35,.72)]
base=set(seg_df_prog.loc[seg_df_prog.C3>7,'id'])
linhas_e7=[]
for j,(a,b) in enumerate(params_prog,1):
    for forma in ['potencial','linear']:
        score=a*np.power(seg_df_prog.area,b) if forma=='potencial' else a+seg_df_prog.area*b
        for limiar in [5,7,9]:
            for filtro in ['todos','arvore']:
                selecionados=set(seg_df_prog.loc[(score>limiar)&
                   ((seg_df_prog.classe_cod==0) if filtro=='arvore' else True),'id'])
                inter=len(selecionados&base); uniao=len(selecionados|base)
                jaccard=inter/uniao if uniao else float('nan')
                for raio in [1,3,5]:
                    prox=pares_df_prog.loc[(pares_df_prog.distancia_m<=raio)&
                                           (pares_df_prog.id.isin(selecionados))]
                    linhas_e7.append({'configuracao':f'C{j}','forma':forma,'limiar':limiar,
                       'filtro':filtro,'distancia_m':raio,'segmentos_elegiveis':len(selecionados),
                       'segmentos_proximos':prox.id.nunique(),'postes_proximos':prox.poste_id.nunique(),
                       'jaccard_vs_C3_potencial_7':jaccard})
e7df=pd.DataFrame(linhas_e7)
assert len(e7df)==216
e7df.to_csv(SAIDA_PROG/'E7_alometria_216_cenarios.csv',index=False)
# Auditoria de qualidade de extrapolação: a base só tem rótulos árvore/grama.
# Uma predição 'arvore' em toda a AOI NÃO prova que o objeto é vegetação.
fracao_arvore_prog = float((seg_df_prog['classe_cod']==0).mean()) if len(seg_df_prog) else float('nan')
qtd_postes_prog = len(POSTES_PROG) if 'POSTES_PROG' in globals() else None
e6_arv1_prog = next((r for r in linhas_e6 if r['filtro']=='arvore' and r['distancia_m']==1),None)
fracao_postes_arv1_prog = (
    float(e6_arv1_prog['postes_proximos']/qtd_postes_prog)
    if e6_arv1_prog and qtd_postes_prog else None
)
avisos_qc_prog = []
if fracao_arvore_prog > .70:
    avisos_qc_prog.append('ALERTA: mais de 70% dos segmentos elegiveis foram previstos como arvore; verificar extrapolacao a fundo sem rotulo de classe nao vegetacao.')
if fracao_postes_arv1_prog is not None and fracao_postes_arv1_prog > .70:
    avisos_qc_prog.append('ALERTA: mais de 70% dos postes foram marcados com segmento previsto como arvore a 1 m; isso NAO constitui indicador validado de risco.')
qcdados_prog = {'fracao_segmentos_previstos_arvore':fracao_arvore_prog,
    'postes_na_aoi':qtd_postes_prog,
    'fracao_postes_com_segmento_arvore_ate_1m':fracao_postes_arv1_prog,
    'avaliacao_de_risco_eletrico_validada':False,
    'classe_nao_vegetacao_rotulada':False,
    'avisos':avisos_qc_prog}
(SAIDA_PROG/'E8_auditoria_qualidade.json').write_text(
    json.dumps(qcdados_prog,indent=2,ensure_ascii=False),encoding='utf-8')
if avisos_qc_prog:
    print('\n*** E8: ALERTA DE QUALIDADE ***')
    for aviso in avisos_qc_prog: print('-',aviso)
    print('Estes resultados sao exploratorios; nao publicar contagens como arvores reais ou risco eletrico.')
manifesto={
  'metodo':'segmentacao por quadrantes Quickshift 256x256, checkpoint/resume; não equivalente ao raster integral',
  'raster_pixels':int(SHAPE_PROG[0]*SHAPE_PROG[1]),
  'total_quadrantes':len(TILES_PROG),
  'quadrantes_validos':sum(caminho_tile_prog(i).exists() for i in range(len(TILES_PROG))),
  'segmentos_validos':len(seg_df_prog),
  'rotulos_sem_conflitos':len(pd.read_csv(SAIDA_PROG/'rotulos_sem_duplicacao.csv')),
  'pares_objeto_poste_5m':len(pares_df_prog),
  'cenarios_e7':len(e7df),
  'controle_qualidade': qcdados_prog,
  'limitacoes':['Limites dos quadrantes podem dividir copas e alterar objetos.',
      'A extração de atributos é por pixel dentro da AOI e em janelas alinhadas aos rasters.',
      'A etapa E2/E3/E4 usa embargo de 25 m entre centroides, não entre polígonos.',
      'Não se usa o cluster 3 histórico para selecionar a base de treinamento.',
      'C1–C6 são índices empíricos sem validação por alturas observadas.',
      'Proximidade horizontal a postes não comprova contato com cabos ou risco elétrico.',
      'Desempenho não pode ser comparado diretamente à acurácia histórica 0,87.'],
  'arquivos_resultados':['E8_auditoria_qualidade.json','E1_auditoria.csv','E2_particoes.csv','E3_E4_comparacoes.csv',
    'E5_kmeans.csv','E5_quickshift_piloto.csv','E6_postes.csv','E7_alometria_216_cenarios.csv']}
(SAIDA_PROG/'E8_manifesto.json').write_text(json.dumps(manifesto,indent=2,ensure_ascii=False),encoding='utf-8')
relatorio = [
'# Execução progressiva — resultados reais da Area1 por quadrantes',
'','**Não é a execução integral do algoritmo original aplicado ao raster inteiro.**','',
f"Quadrantes: {manifesto['quadrantes_validos']} / {manifesto['total_quadrantes']}",
f"Segmentos válidos: {manifesto['segmentos_validos']}",
f"Referências distintas: {manifesto['rotulos_sem_conflitos']}",
f"Cenários E7: {manifesto['cenarios_e7']}",'',
'## E6 — proximidade a postes','',
pd.DataFrame(linhas_e6).to_markdown(index=False),'',
'## Controle de qualidade e limitações','',*['- '+a for a in avisos_qc_prog],*['- '+v for v in manifesto['limitacoes']]]
(SAIDA_PROG/'E8_relatorio.md').write_text('\n'.join(relatorio),encoding='utf-8')
print('ETAPA 4 CONCLUÍDA — confira os CSVs e relatórios em:')
print(SAIDA_PROG)
print('Resultados experimentais por QUADRANTES: não declarar equivalentes à V2 integral histórica.')

## Resultados, rastreabilidade e interpretação

Os **resultados históricos** do método original (**0,87 de acurácia**) estão no [README](README.md). As tabelas de oito algoritmos já publicadas vieram de experimentos derivados documentados; as células deste notebook foram reorganizadas para permitir **uma nova execução independente**, sem misturar esses resultados com as saídas antigas.

**O que deve ser verificado após executar:**

- **Caminho 1:** arquivos `continuo_512_perfil.json`, `quadrantes_256_perfil.json`, `resumo_oito_modelos.csv` e `metricas_mesmos_pontos.csv` devem existir no diretório de saída. O código espera **20 divisões/protocolos e 320 avaliações**; qualquer divergência dos valores históricos deve ser examinada.
- **Caminho 2:** checkpoints dos quadrantes e `comparacao_oito_algoritmos_256.csv` devem estar disponíveis. A etapa de poste/porte é **opcional** e não mede risco elétrico real.
- **Análise dos atributos (E4/E4b), no Caminho 2:** após a Etapa 2, conferir `E4_resumo_ablacao.csv`, `E4_tres_vs_melhor_dupla.csv`, `E4b_oito_modelos_rodadas.csv` e `E4b_oito_modelos_HSV_Canny_vs_tres.csv`. O E4b compara exatamente **brilho HSV + bordas Canny versus NDVI + brilho HSV + bordas Canny** para **oito modelos**. As métricas são desta execução do notebook, com embargo de centroides; não comparar automaticamente às métricas E3c obtidas com embargo de geometrias completas.

**Limitações:** os caminhos com janelas não são o algoritmo global original; a divisão da imagem muda fronteiras de segmentos e, portanto, pode modificar atributos e resultados. O projeto não dispõe de altura medida das árvores, classe de fundo urbano nem geometria dos cabos. Não interprete proximidade a postes como risco elétrico comprovado.
